## Donwload Grassland GPW datasets from
https://zenodo.org/records/15649332 

In [1]:
from pathlib import Path
import rasterio
from rasterio.windows import from_bounds
from rasterio.coords import BoundingBox

# -------------------------------------------------------------------
# CONFIG
# -------------------------------------------------------------------
# Output root
base_path = Path("/home/georg/data/LEON_P5_BII")
out_dir = base_path / "EO_data_raw" / "Grassland_GPW"
out_dir.mkdir(parents=True, exist_ok=True)

# Country bounding boxes: (minx, miny, maxx, maxy) in EPSG:4326
BBOXES = {
    "dnk": (8.076389, 54.559029, 15.193056, 57.751526),      # Denmark
    "nld": (3.360782, 50.723492, 7.227095, 53.554585),       # Netherlands
}

# Yearly URLs (add/remove as needed)
URLS = {
    2018: "https://s3.eu-central-1.wasabisys.com/arco/gpw_grassland_rf.med.filt.bthr_c_30m_20180101_20181231_go_epsg.4326_v2.tif",
    2019: "https://s3.eu-central-1.wasabisys.com/arco/gpw_grassland_rf.med.filt.bthr_c_30m_20190101_20191231_go_epsg.4326_v2.tif",
    2020: "https://s3.eu-central-1.wasabisys.com/arco/gpw_grassland_rf.med.filt.bthr_c_30m_20200101_20201231_go_epsg.4326_v2.tif",
    2021: "https://s3.eu-central-1.wasabisys.com/arco/gpw_grassland_rf.med.filt.bthr_c_30m_20210101_20211231_go_epsg.4326_v2.tif",
    2022: "https://s3.eu-central-1.wasabisys.com/arco/gpw_grassland_rf.med.filt.bthr_c_30m_20220101_20221231_go_epsg.4326_v2.tif",
    2023: "https://s3.eu-central-1.wasabisys.com/arco/gpw_grassland_rf.med.filt.bthr_c_30m_20230101_20231231_go_epsg.4326_v2.tif",
}

# Output filename pattern: e.g., grassland_gpw_2021_dnk.tif
def out_name(year: int, iso3: str) -> Path:
    return out_dir / f"grassland_gpw_{iso3}_{year}.tif"

# -------------------------------------------------------------------
# HELPER
# -------------------------------------------------------------------
def clamp_bbox_to_src(bbox: BoundingBox, src_bounds: BoundingBox) -> BoundingBox | None:
    """
    Intersect a bbox with source bounds and return a clamped bbox.
    Returns None if there is no overlap.
    """
    left   = max(bbox.left,   src_bounds.left)
    right  = min(bbox.right,  src_bounds.right)
    bottom = max(bbox.bottom, src_bounds.bottom)
    top    = min(bbox.top,    src_bounds.top)

    if (left >= right) or (bottom >= top):
        return None
    return BoundingBox(left=left, bottom=bottom, right=right, top=top)

# -------------------------------------------------------------------
# MAIN
# -------------------------------------------------------------------
for year, url in URLS.items():
    # If you get HTTP range-request issues with your GDAL build, prepend /vsicurl/:
    # url = f"/vsicurl/{url}"

    print(f"\n=== Processing year {year} ===")
    with rasterio.Env():  # inherit default GDAL HTTP settings
        with rasterio.open(url) as src:
            src_bounds = src.bounds
            print(f"Source CRS: {src.crs}, bounds: {src_bounds}")

            for iso3, (minx, miny, maxx, maxy) in BBOXES.items():
                bbox = BoundingBox(minx, miny, maxx, maxy)
                clipped_bbox = clamp_bbox_to_src(bbox, src_bounds)

                if clipped_bbox is None:
                    print(f"  [{iso3}] bbox outside raster extent — skipping.")
                    continue

                # Build window from clamped bbox and align to pixels
                window = from_bounds(
                    clipped_bbox.left, clipped_bbox.bottom,
                    clipped_bbox.right, clipped_bbox.top,
                    transform=src.transform
                ).round_offsets().round_lengths()

                # Read all bands in that window
                data = src.read(window=window)

                # Prepare output metadata
                out_meta = src.meta.copy()
                out_meta.update({
                    "height": int(window.height),
                    "width": int(window.width),
                    "transform": src.window_transform(window),
                    "driver": "GTiff",
                    # GeoTIFF creation options for size/performance:
                    "compress": "DEFLATE",
                    "predictor": 2,          # good for continuous rasters
                    "tiled": True,
                    "blockxsize": 512,
                    "blockysize": 512,
                    "BIGTIFF": "IF_SAFER",
                })

                dst_path = out_name(year, iso3)
                with rasterio.open(dst_path, "w", **out_meta) as dst:
                    dst.write(data)

                print(f"  [{iso3}] wrote {dst_path.name}  "
                      f"({out_meta['width']}×{out_meta['height']} px)")


=== Processing year 2018 ===
Source CRS: EPSG:4326, bounds: BoundingBox(left=-179.0005, bottom=-56.000499999982466, right=180.0004999999523, top=76.0005)
  [dnk] wrote grassland_gpw_dnk_2018.tif  (28467×12770 px)
  [nld] wrote grassland_gpw_nld_2018.tif  (15465×11324 px)

=== Processing year 2019 ===
Source CRS: EPSG:4326, bounds: BoundingBox(left=-179.0005, bottom=-56.000499999982466, right=180.0004999999523, top=76.0005)
  [dnk] wrote grassland_gpw_dnk_2019.tif  (28467×12770 px)
  [nld] wrote grassland_gpw_nld_2019.tif  (15465×11324 px)

=== Processing year 2020 ===
Source CRS: EPSG:4326, bounds: BoundingBox(left=-179.0005, bottom=-56.000499999982466, right=180.0004999999523, top=76.0005)
  [dnk] wrote grassland_gpw_dnk_2020.tif  (28467×12770 px)
  [nld] wrote grassland_gpw_nld_2020.tif  (15465×11324 px)

=== Processing year 2021 ===
Source CRS: EPSG:4326, bounds: BoundingBox(left=-179.0005, bottom=-56.000499999982466, right=180.0004999999523, top=76.0005)
  [dnk] wrote grassland_gp